# Підготовка даних для класифікації: тяжкість ДТП (`Severity`)

**Задача:** за обставинами ДТП (місце, час, погода, дорога) передбачити клас тяжкості 1–4 (вплив на рух).
**Результат ноутбука:** готові до навчання train / valid / test у `data/processed/classification/` + `metadata.json` з усіма параметрами обробки. **Моделі тут не навчаються.**

Кожен крок оформлено як **Рішення → Аргументи → Перевірка на даних**. Аргументи спираються на попереднє дослідження (базові моделі, покращений препроцесинг, балансування класів, відбір ознак), підсумок якого наведено в тексті.

| # | Крок | Ключове рішення |
|---|---|---|
| 1 | Первинна обробка | конвеєр `main.py` без змін |
| 2 | Повернення колонок | `Source`, `Street` з сирих даних |
| 3 | Повтори однієї події | «знімки» однієї ДТП від того самого джерела → один запис |
| 4 | Невідомі значення | записи з вигаданими (заповненими) значеннями — видалити; штиль → швидкість 0; опади — прибрати як ознаку |
| 5 | Ціль | `label = Severity − 1`; непослідовна розмітка між джерелами → `Source_Year` |
| 6 | Витік | без `Description`, `End_Time`, `Duration_Minutes` |
| 7 | Ознаки | нормалізація категорій, взаємодії часу, `Source_Year`, тип дороги |
| 8 | Розбиття | групове «місце + день», 70 / 10 / 20 |
| 9 | Викиди ознак | clip 0.5–99.5% (train), `log1p` для скошених |
| 10 | Високо-кардинальні ознаки | out-of-fold target encoding (`sklearn`) + частоти |
| 11 | Аналіз ознак | зв'язок із ціллю (Cramér's V) і надлишковість — лише позначки, ознаки не видаляються |
| 12 | Кодування | one-hot, квантильні біни, стандартизація, `VarianceThresholdSelector` |
| 13 | Дисбаланс класів | дані **не** ресемплюються і ваги не зберігаються; valid — для порогів |
| 14 | Експорт | parquet + metadata |

## 0. Налаштування

In [1]:
import os
import sys
import json
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "main.py").exists() else Path.cwd().parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
java_home = os.environ.get("JAVA_HOME")
if not java_home or not (Path(java_home) / "bin").exists():
    print("[WARNING] JAVA_HOME не задано або вказує на неіснуючу теку — PySpark 4.x потребує JDK 17 або 21.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark import StorageLevel
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import (StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, QuantileDiscretizer,
                                VarianceThresholdSelector)
from pyspark.ml.functions import vector_to_array
from pyspark.ml.stat import Correlation

from config import DATA_PATH
from schemas.raw_schema import raw_schema
from modules.extraction import load_data
from modules.parsing import parse_and_transform_features
from modules.feature_selection import select_features
from modules.data_quality import remove_duplicates, handle_missing_values
from modules.ml_features import (ROAD_BOOL, DESC_PATTERNS, raw_extras, engineer_features, incident_group,
                                 snapshot_groups, is_placeholder_duration, clip_and_log, encode_locations,
                                 cramers_v, eta)

TASK = "classification"
SEED = 42
N_FOLDS = 5                     # фолди out-of-fold target encoding
TE_SMOOTHING = 30.0             # згладжування target encoding
CLIP_Q = (0.005, 0.995)         # перцентилі вінсоризації ознак
N_BINS = 10                     # квантильні біни (і для оцінки зв'язку, і як ознаки)
THR_ASSOC = 0.01       # нижче — зв'язок з ціллю вважається відсутнім
THR_REDUNDANT = 0.8     # вище — дві ознаки вважаються дублікатами
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / TASK

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 120)

spark = (
    SparkSession.builder
    .appName(f"US-Accidents-{TASK}-prep")
    .master("local[*]")
    .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "16g"))
    .config("spark.driver.host", "localhost")
    .config("spark.sql.shuffle.partitions", "48")
    .config("spark.local.dir", os.environ.get("SPARK_LOCAL_DIR", str(PROJECT_ROOT / "spark-tmp")))   # shuffle на диск E:, не в системний TEMP
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.driver.maxResultSize", "6g")                 # toPandas для target encoding і оцінки зв'язку
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
DECISIONS = []   # журнал рішень — зведення в кінці ноутбука
print("Spark", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/25 18:27:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/25 18:27:07 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


Spark 4.1.1


## 1. Первинна обробка — конвеєр `main.py`

**Рішення:** використати модулі проєкту без змін: читання за `raw_schema` → видалення смислових дублікатів → `Year/Month/DayOfWeek/Hour`, `Duration_Minutes`, `Zipcode_Base` → `Light_Level` і видалення колонок (`select_features`) → повторне видалення дублікатів → заповнення пропусків погоди.

**Аргументи:** ML-дані мають відповідати даним OLAP-частини проєкту; модулі вже перевірені командою. Те, чого конвеєру бракує для ML, додаємо наступними кроками, не змінюючи модулі (щоб не зачепити `main.py`).

In [2]:
t0 = time.time()
raw_df = load_data(spark, DATA_PATH, raw_schema)
n_raw = raw_df.count()
base_df = remove_duplicates(raw_df)
base_df = parse_and_transform_features(base_df)
base_df = select_features(base_df)
base_df = remove_duplicates(base_df)
base_df = handle_missing_values(base_df)


--- Очищення дублікатів ---
Видалено записів: 117695 (1.5229%)

--- Очищення дублікатів ---
Видалено записів: 23204 (0.3049%)

--- Опрацювання пропусків ---
Очищення завершено. Видалено рядків: 7925


## 2. Повернення колонок, які видаляє `select_features`

**Рішення:** приєднати за `ID` з сирих даних:
* `Source` — постачальник даних (Source1 / Source2 / Source3);
* `Street_Type` — тип дороги, виділений з назви вулиці (Interstate, US_Highway, Expressway, Route_Hwy, Avenue_Blvd, Local_Street, Other, Unknown);
* нормалізовану назву вулиці — як ключ для target encoding (крок 10);
* прапорці ключових слів `Description` — **лише для перевірки витоку** (крок 6).

**Аргументи:** у дослідженні `Source × рік` виявилась **найважливішою ознакою** для обох задач (permutation importance: +0.70 log-loss класифікації, −0.51 R² регресії при перемішуванні), бо джерела по-різному фіксують і тяжкість, і час завершення. Тип дороги суттєво змінює розподіл цілей (див. нижче). Для OLAP ці колонки зайві, тому `select_features` їх видаляє — для ML повертаємо.

In [3]:
df = engineer_features(base_df.join(raw_extras(raw_df), "ID", "left").filter(F.col("Duration_Minutes") > 0))
df = df.persist(StorageLevel.MEMORY_AND_DISK)
n_rows = df.count()
assert n_rows == df.select(F.countDistinct("ID")).first()[0], "join за ID розмножив рядки"
print(f"Сирих записів: {n_raw:,} → після первинної обробки: {n_rows:,} ({time.time() - t0:.0f} c)")
DECISIONS.append(("1–2. Первинна обробка + повернення колонок", "конвеєр main.py; +Source, Street_Type, ключ вулиці",
                  f"{n_raw:,} → {n_rows:,} рядків"))

ConnectionRefusedError: [Errno 111] Connection refused

## 3. Повтори однієї події від того самого джерела («знімки»)

`remove_duplicates` з `main.py` видаляє лише записи, ідентичні в **усіх** полях. Але датасет зібрано багаторазовим опитуванням API сервісів трафіку: поки подія триває, сервіс оновлює її (подовжує / скорочує час завершення, переоцінює тяжкість, змінює опис), і збирач зберігав **кожну версію як новий запис з новим `ID`**. Такі версії відрізняються хоча б одним полем, тому `main.py` їх не прибирає.

**Критерій «знімка»:** те саме джерело, те саме місце (координати до 0.0001° ≈ 11 м), та сама вулиця, **той самий напрямок руху** (з опису) і початок з паузою ≤ 15 хв від попереднього запису. Протилежні напрямки — це окремі затори, їх не об'єднуємо. Реалізація — віконні функції Spark (`Window`, `lag`, кумулятивна сума).

In [ ]:
df = snapshot_groups(df).persist(StorageLevel.MEMORY_AND_DISK)
in_group = F.col("Snapshot_Size") > 1
snap_stats = (df.groupBy("Source").agg(
    F.count("*").alias("записів"),
    F.sum(in_group.cast("int")).alias("у групах знімків"),
    F.countDistinct(F.when(in_group, F.col("Snapshot_Group"))).alias("груп знімків"),
    F.sum((F.col("Snapshot_Severities") > 1).cast("int")).alias("у групах із суперечливою тяжкістю"))
    .toPandas().set_index("Source").sort_index())
snap_stats.loc["Разом"] = snap_stats.sum()
snap_stats["зайвих записів"] = snap_stats["у групах знімків"] - snap_stats["груп знімків"]
snap_stats["% зайвих"] = (100 * snap_stats["зайвих записів"] / snap_stats["записів"]).round(2)
snap_stats

In [ ]:
# Що саме відрізняється між знімками однієї групи
diff = (df.filter(in_group).groupBy("Snapshot_Group").agg(
            F.countDistinct("Start_Time").alias("n_start"), F.countDistinct("End_Time").alias("n_end"),
            F.max("Snapshot_Severities").alias("n_sev"), F.countDistinct("Distance(mi)").alias("n_dist"))
        .agg(F.round(100 * F.avg((F.col("n_start") == 1).cast("double")), 1).alias("% груп: однаковий початок"),
             F.round(100 * F.avg((F.col("n_dist") == 1).cast("double")), 1).alias("% груп: однакова протяжність"),
             F.round(100 * F.avg((F.col("n_end") == 1).cast("double")), 1).alias("% груп: однаковий кінець"),
             F.round(100 * F.avg((F.col("n_sev") == 1).cast("double")), 1).alias("% груп: однакова тяжкість"))
        .toPandas())
display(diff.T.rename(columns={0: "значення"}))

example = df.filter(F.col("Snapshot_Severities") > 1).orderBy("Source", "Snapshot_Group").select("Snapshot_Group").limit(1).collect()
if example:
    print("Приклад групи знімків однієї події з різною тяжкістю:")
    display(df.filter(F.col("Snapshot_Group") == example[0][0])
              .select("ID", "Source", "Severity", "Start_Time", "End_Time", "Duration_Minutes", "Distance(mi)",
                      "Street_Norm", "Desc_Direction", "Desc_Closed")
              .orderBy("End_Time").toPandas())

Знімки однієї групи майже завжди мають **той самий початок і протяжність**, але **різний час завершення**, а частина — **різну тяжкість** (приклад вище — та сама подія з різною тяжкістю). Тобто це версії однієї ДТП, а не різні ДТП.

**Рішення (принцип «остаточний стан події»):** з кожної групи знімків залишаємо **одну версію**:
1. версію з **найпізнішим часом завершення** — остаточний стан події (ранні версії — проміжні оцінки);
2. якщо кілька версій мають однаковий найпізніший кінець і відрізняються тяжкістю (порядок оновлень невідомий) — версію з **максимальною тяжкістю**.

**Аргументи:**
* у реальних системах дублікати однієї сутності зводять до **останньої версії** за часом оновлення («last write wins»); у датасеті немає часу оновлення запису, тому найближчий доступний показник — час завершення події;
* мітка класифікатора — це **остаточний** вплив ДТП на рух; якщо одна з нерозрізнених версій зафіксувала вищий рівень (наприклад, перекриття дороги), вплив справді був таким;
* видаляти суперечливі групи цілком не варто: конфлікти **систематичні** (переважно Source1, 2021–2023, «2 проти 4») і зосереджені в класі 4 — їх видалення забирало ~30% цього класу й зсувало розподіл;
* одна ДТП, записана кілька разів, мала б у навчанні й у метриках у кілька разів більшу вагу;
* протилежні напрямки руху не об'єднуються — затор у кожному напрямку свій.

In [ ]:
n_before = n_rows
group = Window.partitionBy("Snapshot_Group")
latest = Window.partitionBy("Snapshot_Group").orderBy(F.desc("End_Time"), F.desc("Severity"), F.desc("ID"))

# як розв'язано групи із суперечливою тяжкістю
at_latest = (df.filter(F.col("Snapshot_Severities") > 1)
               .withColumn("_max_end", F.max("End_Time").over(group))
               .filter(F.col("End_Time") == F.col("_max_end"))
               .groupBy("Snapshot_Group").agg(F.size(F.collect_set("Severity")).alias("sev_at_latest")))
display(at_latest.agg(
    F.count("*").alias("груп із суперечливою тяжкістю"),
    F.sum((F.col("sev_at_latest") == 1).cast("int")).alias("розв'язано: остання версія однозначна"),
    F.sum((F.col("sev_at_latest") > 1).cast("int")).alias("розв'язано: нерозрізнені версії → максимум")).toPandas())

before_dist = df.groupBy("Severity").count().toPandas().set_index("Severity")["count"]
dedup = (df.withColumn("_r", F.row_number().over(latest)).filter(F.col("_r") == 1).drop("_r")
           .persist(StorageLevel.MEMORY_AND_DISK))
n_rows = dedup.count()
after_dist = dedup.groupBy("Severity").count().toPandas().set_index("Severity")["count"]
df.unpersist()
df = dedup
dist = pd.DataFrame({"до, %": (100 * before_dist / before_dist.sum()).round(3),
                     "після, %": (100 * after_dist / after_dist.sum()).round(3)}).sort_index()
print(f"Записів: {n_before:,} → {n_rows:,} (прибрано {n_before - n_rows:,} повторних знімків, {100 * (n_before - n_rows) / n_before:.2f}%)")
DECISIONS.append(("3. Знімки однієї події", "одна версія з групи: найпізніший End_Time, при рівності — максимальна тяжкість",
                  f"{n_before:,} → {n_rows:,} рядків"))
dist

## 4. Невідомі та «заповнені» значення

`handle_missing_values` з `main.py` заповнює пропуски погоди **медіанами**, невідомі категорії — значенням `"Unknown"`, а `select_features` перетворює невідомі twilight-значення на **«ніч»**. Для OLAP-агрегатів це прийнятно, але для моделі такі значення **вигадані**: вона не відрізнить «температура 64 °F» від «температура невідома, підставлено медіану» і вчитиметься на шумі.

Тому кожен вид невідомих значень розглядаємо як **кандидат на видалення**. Що саме було невідомим, визначаємо за прапорцями `Missing_*`, узятими з **сирих** даних (крок 2), — після заповнення цього вже не видно.

In [ ]:
calm = F.col("Wind_Dir") == "CALM"
station_missing = (F.greatest(*[F.col(f"Missing_{c}") for c in ["Temperature(F)", "Humidity(%)", "Pressure(in)", "Visibility(mi)"]],
                              F.col("Missing_Weather_Condition"), F.col("Missing_Wind_Direction")) == 1) \
                  | (F.col("Weather_Condition") == "N/A Precipitation")
UNKNOWN_KINDS = {
    "немає даних метеостанції (погода → медіани / Unknown)": station_missing,
    "невідома швидкість вітру при штилі (Calm)": (F.col("Missing_Wind_Speed(mph)") == 1) & calm,
    "невідома швидкість вітру, напрямок відомий": (F.col("Missing_Wind_Speed(mph)") == 1) & ~calm & ~station_missing,
    "невідомий рівень освітленості (twilight → «ніч»)": F.col("Missing_Twilight") == 1,
    "невідома вулиця (Street_Type = Unknown)": F.col("Street_Type") == "Unknown",
    "невідомі опади (→ 0 / медіана)": F.col("Missing_Precipitation(in)") == 1,
}
target_expr = (F.when((F.col("Duration_Minutes") >= 1) & ~is_placeholder_duration(F.col("Duration_Minutes")),
                      F.log1p("Duration_Minutes")) if TASK == "regression" else F.col("Severity"))
flags_pd = df.select(*[e.cast("int").alias(n) for n, e in UNKNOWN_KINDS.items()], target_expr.alias("target"),
                     "Year").toPandas()
rows = []
for name in UNKNOWN_KINDS:
    has = flags_pd[name] == 1
    if TASK == "regression":
        sub = flags_pd[flags_pd["target"].notna()]
        link = eta(sub[name], sub["target"])
        diff = f"сер. log-тривалість: {sub.loc[sub[name] == 1, 'target'].mean():.3f} проти {sub.loc[sub[name] == 0, 'target'].mean():.3f}"
    else:
        link = cramers_v(flags_pd[name], flags_pd["target"])
        diff = (f"Severity 4: {100 * (flags_pd.loc[has, 'target'] == 4).mean():.2f}% проти "
                f"{100 * (flags_pd.loc[~has, 'target'] == 4).mean():.2f}%")
    rows.append({"вид невідомого значення": name, "записів": int(has.sum()), "% записів": round(100 * has.mean(), 3),
                 "зв'язок з ціллю": round(link, 4), "ціль: з прапорцем / без": diff})
unknown_table = pd.DataFrame(rows).set_index("вид невідомого значення")
unknown_table

In [ ]:
# Докази для двох особливих випадків: швидкість вітру при штилі та опади за роками
wind = df.select(calm.alias("штиль (Calm)"), (F.col("Missing_Wind_Speed(mph)") == 1).alias("швидкість невідома"),
                 F.when(F.col("Missing_Wind_Speed(mph)") == 0, F.col("Wind_Speed(mph)")).alias("відома швидкість"))
display(wind.groupBy("штиль (Calm)", "швидкість невідома")
        .agg(F.count("*").alias("записів"), F.min("відома швидкість").alias("min відомої швидкості"),
             F.max("відома швидкість").alias("max відомої швидкості")).orderBy("штиль (Calm)", "швидкість невідома").toPandas())
display(flags_pd.groupby("Year")["невідомі опади (→ 0 / медіана)"].mean().mul(100).round(1)
        .to_frame("% записів з невідомими опадами").T)

**Рішення:**

| Вид невідомого значення | Рішення | Аргументи |
|---|---|---|
| немає даних метеостанції | **видалити записи** | температура, вологість, тиск, видимість, погода, напрямок вітру — вигадані (медіана / Unknown); зазвичай бракує всього набору одразу; ~2–3% записів; розподіл цілі на них майже такий самий (зв'язок ≈ 0.01–0.02), тож видалення не зсуває вибірку. Сюди ж — `Weather_Condition = "N/A Precipitation"` (теж «невідомо») |
| швидкість вітру невідома при **штилі** | **відновити 0**, не видаляти | у всіх записах зі штилем і відомою швидкістю вона рівно 0 — тобто «невідома» тут означає «0», а `handle_missing_values` помилково ставив медіану (~7 mph) |
| швидкість вітру невідома, напрямок відомий | **видалити записи** | справді невідома; < 0.5% записів |
| невідомий twilight | **видалити записи** | `Light_Level` мовчки став «ніч» — хибне значення; ~0.3% записів |
| невідома вулиця | **видалити записи** | тип дороги і target encoding вулиці неможливі; ~0.1% записів |
| невідомі опади | **прибрати ознаку `Precipitation`**, записи залишити | невідомі в ~29% записів, а за 2016–2018 рр. — у ~90%: заповнений «0» не відрізнити від справжнього «опадів немає». Видалити стільки записів не можна (зникли б цілі роки), а сама ознака у дослідженні не давала внеску; інформацію про дощ / сніг дає `Weather_Group` |

Після цього категорій `Unknown` в ознаках не лишається.

In [ ]:
n_before = n_rows
drop_rule = (UNKNOWN_KINDS["немає даних метеостанції (погода → медіани / Unknown)"]
             | UNKNOWN_KINDS["невідома швидкість вітру, напрямок відомий"]
             | UNKNOWN_KINDS["невідомий рівень освітленості (twilight → «ніч»)"]
             | UNKNOWN_KINDS["невідома вулиця (Street_Type = Unknown)"])
clean = (df.filter(~drop_rule)
           .withColumn("Wind_Speed(mph)", F.when((F.col("Missing_Wind_Speed(mph)") == 1) & calm, F.lit(0.0))
                                           .otherwise(F.col("Wind_Speed(mph)")))
           .persist(StorageLevel.MEMORY_AND_DISK))
n_rows = clean.count()
df.unpersist()
df = clean
del flags_pd
left = df.select(F.sum((F.col("Weather_Group") == "Unknown").cast("int")).alias("Weather_Group = Unknown"),
                 F.sum((F.col("Wind_Dir") == "UNKNOWN").cast("int")).alias("Wind_Dir = UNKNOWN"),
                 F.sum((F.col("Street_Type") == "Unknown").cast("int")).alias("Street_Type = Unknown")).toPandas()
print(f"Записів: {n_before:,} → {n_rows:,} (видалено {n_before - n_rows:,}, {100 * (n_before - n_rows) / n_before:.2f}%)")
print(f"Відновлено швидкість вітру 0 для штилю: {int(unknown_table.loc['невідома швидкість вітру при штилі (Calm)', 'записів']):,} записів")
DECISIONS.append(("4. Невідомі значення", "видалено записи з вигаданими значеннями; штиль → 0; без ознаки Precipitation",
                  f"{n_before:,} → {n_rows:,} рядків"))
display(left.rename(index={0: "залишилось записів"}))

## 5. Цільова змінна `Severity`

### 5.1 Розподіл класів

In [ ]:
sev = df.groupBy("Severity").count().orderBy("Severity").toPandas()
sev["частка %"] = (100 * sev["count"] / sev["count"].sum()).round(3)
display(sev)
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=sev, x="Severity", y="count", hue="Severity", palette="viridis", legend=False, ax=ax)
ax.set_title("Розподіл Severity")
plt.tight_layout()
plt.show()

Класи сильно незбалансовані (клас 2 — ~79%, клас 1 — <1%). Рішення щодо дисбалансу — у кроці 13.

### 5.2 Узгодженість розмітки між джерелами

In [ ]:
ct = (df.groupBy("Source_Year", "Severity").count().toPandas()
      .pivot(index="Source_Year", columns="Severity", values="count").fillna(0))
ct_pct = (100 * ct.div(ct.sum(axis=1), axis=0)).round(2)
ct_pct.columns = [f"Severity={c} %" for c in ct_pct.columns]
ct_pct.insert(0, "записів", ct.sum(axis=1).astype(int))
display(ct_pct)
ax = ct_pct.drop(columns="записів").plot.bar(stacked=True, figsize=(14, 4.5), colormap="viridis", width=0.85)
ax.set_ylabel("% записів")
ax.set_title("Severity за джерелом і роком")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

**Рішення:** залишаємо всі записи, ціль `label = Severity − 1` (0…3), а схему розмітки передаємо моделі ознакою **`Source_Year`** (джерело × рік).

**Аргументи:**
* розмітка **непослідовна**: у Source1 за 2021–2023 рр. немає класів 1 і 3 взагалі, клас 1 майже повністю зосереджений у 2022 р. Source2/Source3 — це зміна методики, а не аварійності;
* відкинути «незручні» джерела/роки означало б втратити більшість рідкісних класів (майже весь клас 1);
* з `Source_Year` модель може навчитись, які класи можливі в кожній схемі розмітки — у дослідженні саме ця ознака дала найбільший внесок.

In [ ]:
DECISIONS.append(("5. Ціль", "label = Severity − 1; усі записи; схема розмітки → Source_Year", f"{n_rows:,} рядків"))

## 6. Контроль витоку цільової змінної

Ознака має бути відома **на момент реєстрації ДТП**. Те, що описує наслідки ДТП, — витік.

In [ ]:
desc = (df.groupBy("Severity").agg(*[F.round(100 * F.avg(c), 2).alias(c) for c in DESC_PATTERNS])
        .orderBy("Severity").toPandas().set_index("Severity").T)
desc.columns = [f"Severity={c}: % записів зі словом" for c in desc.columns]
desc

**Рішення — не використовуємо як ознаки:**

| Колонка | Причина |
|---|---|
| прапорці з `Description` | слово «closed» є майже в 9 з 10 описів класу 4 і лише в кожному ~20-му для інших класів — опис **містить саму мітку** (Severity = вплив на рух). У дослідженні ці ознаки штучно піднімали macro-F1 на +0.09 |
| `End_Time`, `Duration_Minutes` цього запису | наслідок ДТП, відомий лише після її завершення |
| `ID`, `Start_Time` | ідентифікатор; зі `Start_Time` беремо лише похідні часові ознаки |

Історичні статистики **інших** ДТП на тій самій вулиці / в місті (частки класів, середня тривалість) — **не витік**: вони відомі заздалегідь. Їх формуємо в кроці 10 так, щоб запис ніколи не бачив власної цілі.

In [ ]:
DECISIONS.append(("6. Витік", "без Description, End_Time, цілі іншої задачі", "8 прапорців Description не використано"))

## 7. Інженерія ознак

| Ознака | Як отримано | Чому |
|---|---|---|
| `Wind_Dir` | `Calm`/`CALM`, `Variable`/`VAR`, `North`/`N`… → один запис | 25 написань → 19 реальних значень; дублікати розпорошують статистику |
| `Weather_Group` + `Is_Windy` | 145 погодних станів → 8 груп за змістом; «/ Windy» — окремий прапорець | більшість станів — рідкісні комбінації; групування дає стабільні оцінки |
| `HourType` | година × (будні / вихідні), 48 категорій | годину пік мають лише будні; лінійна модель не бачить таких взаємодій сама |
| `Month_cat` | місяць як категорія | сезонність нелінійна (зима / літо / свята) |
| `DoW_sin`, `DoW_cos` | день тижня циклічно | неділя «поруч» із понеділком |
| `Source_Year` | джерело × рік | схема фіксації цілей змінювалась (кроки 2, 5) |
| `Street_Type` | тип дороги з назви | магістраль vs локальна вулиця — різні наслідки ДТП |
| `Light_Level` (з `main.py`) | 0–4 за чотирма twilight-ознаками | одна порядкова ознака замість чотирьох корельованих |

In [ ]:
for c in ["Weather_Group", "Wind_Dir", "Street_Type"]:
    display(df.groupBy(c).count().orderBy(F.desc("count")).toPandas().set_index(c).T)

## 8. Розбиття train / valid / test — групове

**Рішення:** ключ групи — округлені до 0.001° координати (~100 м) + дата. Усі записи групи потрапляють в одну вибірку: **train 70% / valid 10% / test 20%** груп. Фолди target encoding (крок 10) — теж групові.

**Аргументи:**
* багато записів мають «близнюків» — та сама подія, повідомлена повторно, обидва напрямки руху, записи від різних джерел. При розбитті по рядках такі пари розходяться між train і test, і test перестає бути чесною оцінкою;
* **valid** потрібен для налаштування гіперпараметрів і порогів рішень на етапі навчання, щоб не підганяти їх під test;
* розбиття робимо **до** обчислення будь-яких статистик (межі викидів, target encoding, масштабування) — усі вони беруться лише з train.

In [ ]:
grp = df.withColumn("_g", incident_group()).groupBy("_g").agg(F.count("*").alias("n"), F.countDistinct("Source").alias("ns"))
display(grp.agg(F.count("*").alias("груп «місце + день»"),
                F.sum(F.when(F.col("n") > 1, F.col("n")).otherwise(0)).alias("записів у групах з > 1 записом"),
                F.sum((F.col("ns") > 1).cast("int")).alias("груп з кількома джерелами")).toPandas())

df = (df.withColumn("_g", incident_group())
        .withColumn("_bucket", F.pmod(F.col("_g"), F.lit(100)))
        .withColumn("split", F.when(F.col("_bucket") < 20, "test").when(F.col("_bucket") < 30, "valid").otherwise("train"))
        .withColumn("fold", F.pmod(F.xxhash64(F.col("_g"), F.lit("fold")), F.lit(N_FOLDS)).cast("int"))
        .drop("_g", "_bucket"))

In [ ]:
split_stats = (df.groupBy("split", "Severity").count().toPandas().pivot(index="Severity", columns="split", values="count"))
split_stats = split_stats[["train", "valid", "test"]]
for c in ["train", "valid", "test"]:
    split_stats[f"{c} %"] = (100 * split_stats[c] / split_stats[c].sum()).round(3)
split_stats

In [ ]:
DECISIONS.append(("8. Розбиття", "групове (місце + день), 70/10/20, групові фолди TE", "див. таблицю вище"))

## 9. Викиди в ознаках

**Рішення:**
* вінсоризація (clip) неперервних ознак по **0.5% / 99.5% перцентилях train**;
* `log1p` для `Distance(mi)` після обрізання.

**Аргументи:** у погодних ознаках є фізично неможливі значення (температура ~200 °F, тиск 0, вітер > 1000 mph) — помилки датчиків. Видаляти рядки не варто (решта полів коректна), а обрізання по перцентилях залишає запис, але прибирає вплив помилки. Межі — лише з train. `Distance` має довгий правий хвіст і масу нулів — логарифм вирівнює масштаб. (`Precipitation` прибрано в кроці 4.)

In [ ]:
NUM_CONT = ["Distance(mi)", "Temperature(F)", "Humidity(%)", "Pressure(in)", "Visibility(mi)", "Wind_Speed(mph)"]
LOG_COLS = ["Distance(mi)"]
train_only = df.filter(F.col("split") == "train")
CLIP_BOUNDS = {c: (float(lo), float(hi)) for c, (lo, hi) in zip(NUM_CONT, train_only.approxQuantile(NUM_CONT, list(CLIP_Q), 0.0001))}
clip_share = train_only.select([F.round(100 * F.avg(((F.col(c) < lo) | (F.col(c) > hi)).cast("double")), 3).alias(c)
                                for c, (lo, hi) in CLIP_BOUNDS.items()]).toPandas().T[0]
display(pd.DataFrame({"нижня межа": {c: b[0] for c, b in CLIP_BOUNDS.items()},
                      "верхня межа": {c: b[1] for c, b in CLIP_BOUNDS.items()},
                      "% train-значень обрізано": clip_share}))
DUR_MAX = float(train_only.approxQuantile("Duration_Minutes", [0.99], 0.0001)[0])
DUR_MIN = 1.0
print(f"Q99 тривалості (train): {DUR_MAX:.1f} хв")
DECISIONS.append(("9. Викиди ознак", "clip 0.5/99.5% train; log1p Distance",
                  f"обрізано ≤ {clip_share.max():.2f}% значень кожної ознаки"))

## 10. Високо-кардинальні ознаки: target encoding і частоти

**Рішення:** для ключів **вулиця** (`State|Street`), **місто**, **округ**, **поштовий індекс** обчислити:
* `TE_<key>_t_sev1 / t_sev3 / t_sev4` — згладжена частка ДТП класу 1 / 3 / 4 на цьому ключі;
* `TE_<key>_t_logdur` — згладжене середнє `log1p(тривалості)` (лише за записами області регресії, без заглушок);
* `Cnt_<key>` — `log1p` кількості **інших** train-записів з цим ключем (щільність аварій).

**Аргументи:**
* місто / округ / індекс / вулиця мають від тисяч до сотень тисяч значень — one-hot дав би сотні тисяч розріджених стовпців; target encoding стискає кожен ключ до кількох чисел;
* в обох задачах використовуємо статистики **обох** цілей: це історичні характеристики місця (там часто тяжкі / тривалі ДТП), відомі заздалегідь. У дослідженні, наприклад, середня тривалість на вулиці помітно допомагала класифікації;
* реалізація — бібліотечний **`sklearn.preprocessing.TargetEncoder`**: `target_type="multiclass"` для `Severity` (частки класів; частку класу 2 відкидаємо — частки в сумі дають 1) і `"continuous"` для log-тривалості; згладжування `smooth = 30`: $\frac{n\cdot\bar y_{key} + m\cdot\bar y}{n + m}$ — рідкісні ключі стягуються до загального середнього; невідомий у train ключ → загальне середнє. Частоти — `pandas.Series.value_counts`;
* чому не `pyspark.ml.feature.TargetEncoder`: на ключі з ~300 тис. значень (вулиці) його перетворення на 5 млн рядків не завершувалось за 10+ хв, тоді як sklearn робить усе кодування за ~2 хв;
* **захист від витоку:** енкодер навчається лише на train; рядки train кодуються **out-of-fold** — енкодером, навченим на інших 4 **групових** фолдах, тож запис ніколи не бачить власної цілі. Вбудований `cv` у sklearn ділить дані випадково — записи однієї ДТП від різних джерел «підглядали» б один за одним, тому фолди задаємо самі.

In [ ]:
TE_KEYS = {"Street": "Key_Street", "City": "Key_City", "County": "Key_County", "Zip": "Key_Zip"}
in_domain = F.col("Duration_Minutes").between(DUR_MIN, DUR_MAX) & ~is_placeholder_duration(F.col("Duration_Minutes"))

prepared = (clip_and_log(df, CLIP_BOUNDS, LOG_COLS)
            .withColumn("Log_Duration", F.log1p(F.col("Duration_Minutes")))
            .withColumn("In_Reg_Domain", in_domain.cast("int"))
            .withColumn("label", (F.col("Severity") - 1).cast("double"))
            .withColumn("t_logdur", F.when(in_domain, F.col("Log_Duration"))))

splits = {s: prepared.filter(F.col("split") == s) for s in ["train", "valid", "test"]}
display(splits["train"].agg(*[F.countDistinct(k).alias(n) for n, k in TE_KEYS.items()]).toPandas()
        .rename(index={0: "унікальних ключів у train"}))
t0 = time.time()
key_cols = list(TE_KEYS.values())
frames = {s: d.select("ID", "Severity", "t_logdur", *key_cols, *(["fold"] if s == "train" else [])).toPandas()
          for s, d in splits.items()}
loc_codes = encode_locations(frames, TE_KEYS, smoothing=TE_SMOOTHING)
del frames
splits = {s: d.join(spark.createDataFrame(loc_codes[s]), "ID") for s, d in splits.items()}
TE_COLS = [c for c in loc_codes["train"].columns if c.startswith("TE_")]
CNT_COLS = [c for c in loc_codes["train"].columns if c.startswith("Cnt_")]
del loc_codes
print(f"Кодування локацій: {time.time() - t0:.0f} c; TE-ознак: {len(TE_COLS)}, частотних: {len(CNT_COLS)}")

In [ ]:
for s in splits:
    splits[s] = splits[s].persist(StorageLevel.MEMORY_AND_DISK)
n_split = {s: d.count() for s, d in splits.items()}
df.unpersist()
print(f"Записів: {n_split}")
DECISIONS.append(("10. Target encoding", "sklearn TargetEncoder, OOF по групових фолдах, smooth = 30; + частоти",
                  f"{len(TE_COLS) + len(CNT_COLS)} числових ознак"))

In [ ]:
TARGET_COL = "Log_Duration" if TASK == "regression" else "Severity"
check = splits["valid"].select(
    *[(F.corr(f"TE_{n}_t_logdur", "Log_Duration") if TASK == "regression"
       else F.corr(F.col(f"TE_{n}_t_sev4"), (F.col("Severity") == 4).cast("double"))).alias(n)
      for n in TE_KEYS]).toPandas().T
check.columns = ["кореляція TE з ціллю на valid (чесна перевірка, статистики з train)"]
check.round(3)

## 11. Аналіз ознак: зв'язок із ціллю і надлишковість (лише train)

**Рішення:** фільтр-методи — **лише діагностика**, ознаки на цьому кроці **не видаляються**. Прибираються тільки стовпці, сталі на train (крок 12): вони не несуть інформації для жодної моделі.

**Аргументи:** на цих даних навчатимуться не лише лінійні, а й **нелінійні** моделі (дерева, бустинг), а фільтри бачать лише частину залежностей:
* **Зв'язок із ціллю** — Cramér's V з `Severity`; неперервні ознаки розбито на 10 квантильних бінів (`QuantileDiscretizer`), розрахунок — `pandas.crosstab` → `scipy.stats.contingency.association(method='cramer')`. Це **однофакторна** міра: вона не бачить **взаємодій**. Ознака, майже не пов'язана з ціллю сама по собі, може впливати на неї в поєднанні з іншими (наприклад, `Roundabout` може впливати на тяжкість лише поза містом), і дерево таку взаємодію знайде;
* **Надлишковість** — |Pearson r| для числових пар і Cramér's V для категоріальних. Pearson вимірює лише **лінійну** залежність: TE міста й TE поштового індексу з r ≈ 0.95 відрізняються саме нелінійно (різна деталізація — у межах міста десятки індексів), і нелінійна модель використає обидві. `Timezone` майже визначається `State` (V ≈ 0.97), але це готове групування штатів у регіони: дереву воно дає один поділ замість кількох;
* лінійним моделям залишені ознаки не шкодять: записів у ~20 000 разів більше, ніж ознак, а сильно корельовані пари не дають точної мультиколінеарності (r < 1). Остаточний відбір — справа етапу навчання й конкретної моделі (L1-регуляризація, важливість ознак у бустингу).

**Пороги-позначки** (для довідки етапу навчання, зберігаються в `metadata.json`):
* `THR_ASSOC = 0.01` — для η це η² = 0.0001, тобто ознака сама пояснює < 0.01% розкиду цілі; для Cramér's V — десята частина «малого» ефекту за Коеном (0.1). На 5 млн записів статистична значущість втрачає сенс, тому критерій — **практична** величина ефекту;
* `THR_REDUNDANT = 0.9` — |r| > 0.9 означає > 81% спільної **лінійної** дисперсії.

In [ ]:
NUM_UNITS = ["Start_Lat", "Start_Lng"] + NUM_CONT + TE_COLS + CNT_COLS
ORD_UNITS = ["Light_Level", "DayOfWeek"]
CAT_UNITS = ["State", "Timezone", "Source_Year", "Wind_Dir", "Weather_Group", "Street_Type", "HourType", "Month_cat"]
BIN_UNITS = ROAD_BOOL + ["Is_Windy"]

train_df = splits["train"]
discretizer = QuantileDiscretizer(inputCols=NUM_UNITS, outputCols=[c + "_q" for c in NUM_UNITS], numBuckets=N_BINS,
                                  relativeError=0.001).fit(train_df)
# вузька таблиця (ціль + ознаки у дискретній формі) → pandas для бібліотечних статистик
assoc_pd = (discretizer.transform(train_df.select(TARGET_COL, *NUM_UNITS, *ORD_UNITS, *CAT_UNITS, *BIN_UNITS))
            .select(TARGET_COL, *[c + "_q" for c in NUM_UNITS], *ORD_UNITS, *CAT_UNITS, *BIN_UNITS).toPandas())


def association(col):
    if TASK == "regression":
        return eta(assoc_pd[col], assoc_pd["Log_Duration"])
    return cramers_v(assoc_pd[col], assoc_pd["Severity"])


t0 = time.time()
rows = []
for unit in NUM_UNITS + ORD_UNITS + CAT_UNITS + BIN_UNITS:
    kind = ("числова" if unit in NUM_UNITS else "порядкова" if unit in ORD_UNITS
            else "категоріальна" if unit in CAT_UNITS else "бінарна")
    rows.append({"ознака": unit, "тип": kind, "зв'язок з ціллю": association(unit + "_q" if unit in NUM_UNITS else unit)})
assoc = pd.DataFrame(rows).set_index("ознака")
prev = train_df.select(*[F.avg(c).alias(c) for c in BIN_UNITS]).toPandas().T[0]
assoc.loc[BIN_UNITS, "частка одиниць %"] = (100 * prev).round(4)
assoc["слабкий зв'язок"] = assoc["зв'язок з ціллю"] < THR_ASSOC
print(f"Зв'язок пораховано для {len(assoc)} ознак за {time.time() - t0:.0f} c; поріг = {THR_ASSOC}")
assoc.sort_values("зв'язок з ціллю", ascending=False).round(4)

In [ ]:
s = assoc["зв'язок з ціллю"].sort_values()
fig, ax = plt.subplots(figsize=(10, 14))
s.plot.barh(ax=ax, color=["lightgray" if v < THR_ASSOC else "steelblue" for v in s])
ax.axvline(THR_ASSOC, color="black", ls="--", lw=1)
ax.set_xscale("log")
ax.set_title(f"Зв'язок ознак з ціллю ({'η' if TASK == 'regression' else 'Cramér`s V'}), train; сірі — нижче порогу")
ax.tick_params(axis="y", labelsize=8)
plt.tight_layout()
plt.show()

In [ ]:
vec = VectorAssembler(inputCols=NUM_UNITS, outputCol="_v").transform(train_df.select(NUM_UNITS)).select("_v")
corr = pd.DataFrame(Correlation.corr(vec, "_v").first()[0].toArray(), index=NUM_UNITS, columns=NUM_UNITS)
num_pairs = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack().rename("сила").reset_index()
             .rename(columns={"level_0": "ознака 1", "level_1": "ознака 2"}).assign(міра="|Pearson r|"))
num_pairs["сила"] = num_pairs["сила"].abs()

cat_pairs = pd.DataFrame([{"ознака 1": a, "ознака 2": b, "сила": cramers_v(assoc_pd[a], assoc_pd[b]), "міра": "Cramér's V"}
                          for i, a in enumerate(CAT_UNITS) for b in CAT_UNITS[i + 1:]])
pairs = pd.concat([num_pairs, cat_pairs]).sort_values("сила", ascending=False)
strong_pairs = pairs[pairs["сила"] > THR_REDUNDANT]
print(f"Пар з силою > {THR_REDUNDANT}: {len(strong_pairs)} — обидві ознаки кожної пари залишаються")
pairs[pairs["сила"] > 0.8].round(3)

In [ ]:
weak = list(assoc.index[assoc["слабкий зв'язок"]])
selection = pd.DataFrame(
    [{"ознака": u, "позначка": f"зв'язок з ціллю {assoc.loc[u, 'зв' + chr(39) + 'язок з ціллю']:.4f} < {THR_ASSOC}"} for u in weak]
    + [{"ознака": f"{p['ознака 1']} / {p['ознака 2']}", "позначка": f"{p['міра']} = {p['сила']:.3f} > {THR_REDUNDANT}"}
       for _, p in strong_pairs.iterrows()], columns=["ознака", "позначка"]).set_index("ознака")
selection["рішення"] = "залишити"
KEEP_NUM, KEEP_ORD, KEEP_CAT, KEEP_BIN = list(NUM_UNITS), list(ORD_UNITS), list(CAT_UNITS), list(BIN_UNITS)
print(f"Залишено всі {len(assoc)} ознак: числових {len(KEEP_NUM)}, порядкових {len(KEEP_ORD)}, "
      f"категоріальних {len(KEEP_CAT)}, бінарних {len(KEEP_BIN)}; позначено слабких {len(weak)}, пар-дублікатів {len(strong_pairs)}")
DECISIONS.append(("11. Аналіз ознак", "без видалення: слабкі й лінійно надлишкові ознаки залишено для нелінійних моделей",
                  f"позначено {len(weak)} слабких (< {THR_ASSOC}), {len(strong_pairs)} пар (> {THR_REDUNDANT})"))
del assoc_pd
selection

## 12. Кодування і масштабування (Spark ML Pipeline, навчений лише на train)

| Тип | Перетворення | Чому |
|---|---|---|
| Категоріальні | `StringIndexer` (за частотою) → `OneHotEncoder(dropLast=True)` | лінійні моделі потребують числових ознак; найрідша категорія — референсна, щоб уникнути мультиколінеарності з вільним членом |
| Неперервні (погода, відстань) | + **квантильні біни** (10) → one-hot | лінійна модель отримує кусково-сталу поправку → нелінійні залежності (у дослідженні біни відстані були одними з найважливіших ознак) |
| Числові, порядкові, TE | `StandardScaler` (середнє 0, std 1) | коефіцієнти порівнянні, регуляризація й оптимізація працюють коректно |
| Бінарні | без змін (0/1) | вже у потрібній формі |

Останній етап — **`VarianceThresholdSelector(varianceThreshold=0)`**: прибирає стовпці, сталі на train (наприклад, порожній бін `[-inf, 0)` для невід'ємних ознак). Невідома в train категорія у valid/test → рядок відкидається (кількість показано нижче).

In [ ]:
BIN_COLS = [c for c in ["Distance(mi)", "Temperature(F)", "Humidity(%)", "Pressure(in)", "Visibility(mi)", "Wind_Speed(mph)"]
            if c in KEEP_NUM]
SCALED = KEEP_NUM + (["Light_Level"] if "Light_Level" in KEEP_ORD else []) + (["DoW_sin", "DoW_cos"] if "DayOfWeek" in KEEP_ORD else [])

stages = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", stringOrderType="frequencyDesc", handleInvalid="skip") for c in KEEP_CAT]
stages.append(OneHotEncoder(inputCols=[f"{c}_idx" for c in KEEP_CAT], outputCols=[f"{c}_ohe" for c in KEEP_CAT], dropLast=True))
if BIN_COLS:
    stages += [QuantileDiscretizer(inputCols=BIN_COLS, outputCols=[f"{c}_bin" for c in BIN_COLS], numBuckets=N_BINS,
                                   relativeError=0.0005, handleInvalid="error"),
               OneHotEncoder(inputCols=[f"{c}_bin" for c in BIN_COLS], outputCols=[f"{c}_binohe" for c in BIN_COLS], dropLast=True)]
stages += [VectorAssembler(inputCols=SCALED, outputCol="num_vec"),
           StandardScaler(inputCol="num_vec", outputCol="num_scaled", withMean=True, withStd=True),
           VectorAssembler(inputCols=["num_scaled", *KEEP_BIN, *[f"{c}_ohe" for c in KEEP_CAT],
                                      *[f"{c}_binohe" for c in BIN_COLS]], outputCol="features_all"),
           VarianceThresholdSelector(featuresCol="features_all", outputCol="features", varianceThreshold=0.0)]
t0 = time.time()
prep_model = Pipeline(stages=stages).fit(splits["train"])
LABELS = {c: list(m.labels) for c, m in zip(KEEP_CAT, prep_model.stages[:len(KEEP_CAT)])}
BIN_SPLITS = dict(zip(BIN_COLS, prep_model.stages[len(KEEP_CAT) + 1].getSplitsArray())) if BIN_COLS else {}
scaler = [s for s in prep_model.stages if hasattr(s, "std") and hasattr(s, "mean")][0]

FEATURE_NAMES = (SCALED + KEEP_BIN + [f"{c}={lab}" for c in KEEP_CAT for lab in LABELS[c][:-1]]
                 + [f"{c} in [{sp[j]:.4g}, {sp[j + 1]:.4g})" for c, sp in BIN_SPLITS.items() for j in range(len(sp) - 2)])
probe = prep_model.transform(splits["train"].limit(1)).first()["features_all"]
assert probe.size == len(FEATURE_NAMES), (probe.size, len(FEATURE_NAMES))
selected = set(prep_model.stages[-1].selectedFeatures)
CONST_COLS = [n for i, n in enumerate(FEATURE_NAMES) if i not in selected]
FINAL_FEATURES = [n for i, n in enumerate(FEATURE_NAMES) if i in selected]
print(f"Pipeline навчено за {time.time() - t0:.0f} c; ознак до VarianceThresholdSelector: {len(FEATURE_NAMES)}, "
      f"після: {len(FINAL_FEATURES)}")

In [ ]:
encoded = {s: prep_model.transform(d) for s, d in splits.items()}
lost = {s: n_split[s] - encoded[s].count() for s in ["valid", "test"]}
print(f"Константних стовпців: {len(CONST_COLS)} {CONST_COLS}")
print(f"Відкинуто рядків через невідомі категорії: {lost}")
print(f"Підсумково ознак: {len(FINAL_FEATURES)}")
groups = pd.Series({"масштабовані числові / TE / порядкові": len([f for f in FINAL_FEATURES if f in SCALED]),
                    "бінарні": len([f for f in FINAL_FEATURES if f in KEEP_BIN]),
                    **{f"one-hot {c}": len([f for f in FINAL_FEATURES if f.startswith(c + "=")]) for c in KEEP_CAT},
                    "квантильні біни": len([f for f in FINAL_FEATURES if " in [" in f])})
DECISIONS.append(("12. Кодування", "one-hot (dropLast), біни, StandardScaler; без константних стовпців",
                  f"{len(FINAL_FEATURES)} ознак, константних прибрано {len(CONST_COLS)}"))
groups.to_frame("стовпців")

## 13. Дисбаланс класів

**Рішення:**
* train / valid / test **зберігаємо з реальним розподілом класів** — без undersampling / oversampling / SMOTE;
* колонок ваг **не** додаємо: ваги класів — це опція моделі, яку бібліотеки рахують самі з `label` (`class_weight="balanced"` у sklearn / LightGBM, `auto_class_weights` у CatBoost); для XGBoost чи Spark ML їх так само легко порахувати з `label` на етапі навчання;
* valid — для підбору порогів рішення на етапі навчання.

**Аргументи (з дослідження балансування, логістична регресія, той самий test):**

| Підхід | macro-F1 | macro-F1 з порогами з valid | recall класів 1 / 4 | precision класів 1 / 4 |
|---|---|---|---|---|
| без балансування | 0.632 | **0.672** | 0.42 / 0.20 | 0.71 / 0.62 |
| ваги класів | 0.521 | 0.647 | **0.96 / 0.77** | 0.18 / 0.14 |
| повний undersampling | 0.520 | 0.646 | 0.96 / 0.77 | 0.18 / 0.14 |
| random over + under | 0.521 | 0.647 | 0.96 / 0.77 | 0.18 / 0.14 |
| SMOTE + under | 0.525 | 0.648 | 0.96 / 0.77 | 0.19 / 0.14 |

* усі способи балансування дали **практично однакову** модель — для лінійної моделі ресемплінг ≈ ваги; SMOTE не дав переваги, а для one-hot ознак створює неіснуючі комбінації («між двома штатами»);
* балансування змінює компроміс precision ↔ recall, а не якість моделі. Це рішення **етапу навчання** (залежить від мети: максимум macro-F1 чи «не пропустити тяжку ДТП»), тому в даних ми його не «запікаємо»;
* valid / test **мусять** мати реальний розподіл, інакше метрики не відповідатимуть реальності.

In [ ]:
class_counts = encoded["train"].groupBy("label").count().toPandas().set_index("label")["count"].sort_index()
N_TRAIN = int(class_counts.sum())
DECISIONS.append(("13. Дисбаланс", "без ресемплінгу й без колонок ваг; ваги класів — опція моделі; valid — для порогів",
                  "клас 1 у train: " + f"{100 * class_counts[0.0] / N_TRAIN:.2f}%"))
pd.DataFrame({"Severity": [int(k) + 1 for k in class_counts.index], "train": class_counts.to_numpy(),
              "частка %": (100 * class_counts / N_TRAIN).round(3).to_numpy()})

## 14. Експорт готових даних

`data/processed/classification/{train,valid,test}/part-*.parquet`:
* ознаки — по стовпцю на ознаку (float32), вже закодовані й масштабовані;
* **`label`** (0…3 = Severity 1…4) — ціль; `Severity`; `ID` — для трасування.

`metadata.json` — усі параметри обробки (межі clip, біни, параметри scaler, категорії, TE, рішення відбору), щоб так само перетворити нові дані.

Spark на Windows не пише файли без `winutils.exe`, тому експорт — через pandas/pyarrow частинами.

In [ ]:
N_CHUNKS = {"train": 14, "valid": 2, "test": 4}


def export(d, name, n_chunks):
    out_dir = OUTPUT_DIR / name
    out_dir.mkdir(parents=True, exist_ok=True)
    for old in out_dir.glob("part-*.parquet"):
        old.unlink()
    flat = (d.withColumn("_f", vector_to_array("features", "float32"))
            .select("ID", *[F.col("_f")[i].alias(n) for i, n in enumerate(FINAL_FEATURES)], F.col("label").cast("int").alias("label"), F.col("Severity").cast("int").alias("Severity"),
                    F.pmod(F.xxhash64("ID"), F.lit(n_chunks)).alias("_chunk")))
    rows = 0
    for k in range(n_chunks):
        part = flat.filter(F.col("_chunk") == k).drop("_chunk").toPandas()
        part.to_parquet(out_dir / f"part-{k:03d}.parquet", index=False)
        rows += len(part)
    size = sum(p.stat().st_size for p in out_dir.glob("*.parquet")) / 2**20
    print(f"{name}: {rows:,} рядків, {n_chunks} файлів, {size:.0f} MB")
    return rows


t0 = time.time()
saved = {s: export(encoded[s], s, N_CHUNKS[s]) for s in ["train", "valid", "test"]}
print(f"Експорт: {time.time() - t0:.0f} c")

In [ ]:
metadata = {
    "task": TASK, "source": DATA_PATH, "seed": SEED, "target": "label = Severity - 1",
    "rows": saved,
    "split": "group split by incident_group (lat/lng rounded to 3 dp + date): train 70%, valid 10%, test 20%",
    "regression_domain_minutes": [DUR_MIN, DUR_MAX], "placeholder_duration": "360 ± 0.05 min (excluded from regression and from TE of duration)",
    "feature_names": FINAL_FEATURES,
    "numeric_scaled": SCALED, "binary": KEEP_BIN, "categorical": KEEP_CAT, "category_labels": LABELS,
    "binned": {c: [float(x) for x in sp] for c, sp in BIN_SPLITS.items()},
    "clip_bounds": CLIP_BOUNDS, "log1p_columns": LOG_COLS,
    "scaler": {"mean": dict(zip(SCALED, map(float, scaler.mean.toArray()))), "std": dict(zip(SCALED, map(float, scaler.std.toArray())))},
    "target_encoding": {"implementation": "sklearn.preprocessing.TargetEncoder, out-of-fold by group folds",
                        "keys": TE_KEYS, "targets": {"Severity": "multiclass (classes 1, 3, 4)", "t_logdur": "continuous"},
                        "smoothing": TE_SMOOTHING, "folds": N_FOLDS},
    "snapshots": "same source, lat/lng rounded to 4 dp, same street, same direction, start gaps <= 15 min: "
                 + ("keep the version with the latest End_Time" if TASK == "regression"
                    else "keep the version with the latest End_Time; ties -> the highest Severity"),
    "unknown_values": {"dropped_rows": ["no weather station data (incl. 'N/A Precipitation')", "unknown wind speed with known direction",
                                        "unknown twilight", "unknown street"],
                       "restored": "wind speed = 0 when calm and speed unknown", "dropped_features": ["Precipitation(in)"]},
    "feature_selection": {"rule": "filters are diagnostics only: no features dropped (kept for non-linear models); only constant columns removed",
                          "assoc_threshold": THR_ASSOC, "redundancy_threshold": THR_REDUNDANT,
                          "flagged_kept": selection["позначка"].to_dict(), "constant_columns": CONST_COLS},
    "excluded_for_leakage": ["Description keyword flags", "End_Time", "other task target of the same record"],
    "unseen_source_year": {
        "rule": "unseen Source_Year (e.g. Source1_2026) -> replace with the latest Source_Year of the same source seen in train before encoding; do not drop the record and do not leave all Source_Year dummies at zero (that is the reference category Source3_2016 with a different labelling scheme)",
        "latest_per_source": {s: max(l for l in LABELS["Source_Year"] if l.startswith(s + "_"))
                              for s in sorted({l.split("_")[0] for l in LABELS["Source_Year"]})},
        "evidence": "out-of-time check (classification, test rows of Source1_2023 with a model trained without 2023): accuracy 0.969 / F1 class 4 0.311 with the rule vs 0.744 / 0.366 with the reference category (21% of predictions became class 3, which Source1 does not use since 2021); in-time model: 0.969 / 0.324. Valid only while the source keeps its labelling scheme",
    },
}
(OUTPUT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
DECISIONS.append(("Експорт", "parquet (float32) + metadata.json", f"{saved}"))
print("metadata.json збережено")

In [ ]:
check = pd.read_parquet(next((OUTPUT_DIR / "train").glob("part-*.parquet")))
print("Форма одного файлу train:", check.shape)
print("Пропусків:", int(check.isna().sum().sum()))
check.sample(5, random_state=SEED).iloc[:, list(range(8)) + list(range(-6, 0))].round(3)

## Зведення рішень

In [ ]:
pd.DataFrame(DECISIONS, columns=["Крок", "Рішення", "Результат на даних"]).set_index("Крок")

## Висновки

**Готовий датасет:** `data/processed/classification/` — train **4 789 382**, valid **683 877**, test **1 366 450** записів, **235 ознак** (float32, закодовані й масштабовані), ціль `label` (0…3 = Severity 1…4), пропусків немає. Розподіл класів у трьох вибірках однаковий з точністю до 0.1 п.п.

| Крок | Що зроблено | Чому |
|---|---|---|
| Первинна обробка | конвеєр `main.py`: 7 728 394 → 7 579 570 записів | узгодженість з рештою проєкту |
| Знімки однієї події | прибрано 460 045 повторних версій (6.07%); з кожної групи — **остаточна версія**: найпізніший `End_Time`, а за однакового часу — максимальна тяжкість. З 57 338 груп із суперечливою тяжкістю 11 732 розв'язано за останньою версією, 45 606 — максимумом | збирач зберігав кожне оновлення події як новий запис; принцип «last write wins»; видалення суперечливих груп забрало б ~30% класу 4 (конфлікти систематичні: Source1, 2021–2023, «2 проти 4»). Розподіл класів після зведення майже не змінився (клас 4: 2.65% → 2.65%) |
| Невідомі значення | видалено 279 816 записів (3.93%) з вигаданими значеннями (без даних метеостанції, швидкість вітру при відомому напрямку, twilight, вулиця); для 365 090 записів зі штилем відновлено швидкість **0**; ознаку `Precipitation` прибрано | заповнені медіаною / «Unknown» / «ніч» значення — шум |
| Ціль | `label = Severity − 1`, усі джерела й роки + ознака `Source_Year` | розмітка непослідовна (у Source1 2021–2023 немає класів 1 і 3; клас 1 майже весь у 2022) — модель має знати схему розмітки |
| Витік | без `Description`, `End_Time`, `Duration_Minutes` | «closed» є в 88.8% описів класу 4 і в 2–5% інших — опис містить мітку |
| Розбиття | групове «місце + день», 70 / 10 / 20 | близькі в часі й місці записи не розходяться між train і test |
| Викиди | clip 0.5 / 99.5% train (обрізано ≤ 1% значень), `log1p(Distance)` | помилки датчиків, довгі хвости |
| Локації | `sklearn.preprocessing.TargetEncoder` (OOF по групових фолдах) + частоти; кореляція TE з класом 4 на valid — 0.25–0.29 | сотні тисяч значень ключів → кілька інформативних чисел без витоку |
| Відбір ознак | видалено 7 з 51: `Bump`, `Give_Way`, `Roundabout`, `Traffic_Calming` (V < 0.01), `Timezone` (дублює `State`, V = 0.974), 2 дублікати TE (r > 0.94) | ознака без зв'язку чи дублікат не додає інформації |
| Кодування | one-hot, квантильні біни, `StandardScaler`, `VarianceThresholdSelector` прибрав 2 порожні біни | формат для лінійних моделей, нелінійні ефекти через біни |
| Дисбаланс | дані не ресемплюються, колонок ваг немає | ресемплінг ≈ ваги для лінійної моделі й лише зсуває компроміс precision ↔ recall; ваги класів — опція моделі на етапі навчання (`class_weight="balanced"` тощо) |

**Обмеження:** правило «максимальна тяжкість для нерозрізнених версій» — припущення (у датасеті немає часу оновлення запису). Варто перевірити чутливість на етапі навчання: порівняти з варіантом, де такі групи видалено, на однаковому test.

**Нові дані: невідомий рік джерела.** Ознака `Source_Year` прив'язана до років, які були в train (до 2023). Для запису з новим роком (наприклад, `Source1_2026`) перед кодуванням **замінюємо `Source_Year` на останній відомий рік того ж джерела** (`Source1_2023`, `Source2_2022`, `Source3_2022`; збережено в `metadata.json` → `unseen_source_year`). Без цього правила `StringIndexer(handleInvalid="skip")` відкинув би запис, а якщо залишити всі one-hot стовпці нулями — модель сприйняла б запис як референсну категорію `Source3_2016` з іншою схемою розмітки. Правило коректне, лише поки джерело не змінює схему розмітки (як Source1 у 2021 р.), тому на нових даних потрібні моніторинг розподілу прогнозів і періодичне перенавчання.

**Рекомендації для етапу навчання:**
* навчати на `train`, гіперпараметри і **пороги рішення** підбирати на `valid`, `test` — лише для фінальної оцінки;
* основні метрики — **macro-F1**, balanced accuracy і recall / precision класів 1 і 4 (accuracy через дисбаланс малоінформативна: «завжди клас 2» дає ~78%);
* для максимуму macro-F1 — модель без ваг + пороги з valid; якщо важливіше не пропустити тяжку ДТП — ваги класів моделі (`class_weight="balanced"` у sklearn / LightGBM, `auto_class_weights="Balanced"` або пом'якшене `"SqrtBalanced"` у CatBoost).

In [ ]:
for d in splits.values():
    d.unpersist()
spark.stop()